In [ ]:
# =========================================================
# Install PyTorch Geometric
# =========================================================

!pip install torch-geometric pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html


# =========================================================
# Check Kaggle Input Files
# =========================================================

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# =========================================================
# Base Path
# =========================================================

BASE_PATH = "/kaggle/input/datasets/drrajgauravmishra/spatiotemporal-gnn/"


# =========================================================
# Imports
# =========================================================

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GATv2Conv

import numpy as np
import random
import json


# =========================================================
# GPU Configuration
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


# =========================================================
# Configuration
# =========================================================

seeds = [42, 123, 2023]

epochs = 1000
hidden_channels = 64
heads = 4

learning_rate = 0.0005

years = range(2000, 2024)


# =========================================================
# Load Static Graph
# =========================================================

data = torch.load(
    BASE_PATH + "static_spatial_graph.pt",
    weights_only=False
)

data.edge_index = data.edge_index.to(device)

print("Graph loaded successfully")


# =========================================================
# GLOBAL NORMALIZATION (ALL YEARS)
# =========================================================

all_years_data = []

for year in years:

    yearly_data = torch.load(
        BASE_PATH + f"node_features_{year}.pt"
    )

    yearly_data = torch.nan_to_num(yearly_data)

    all_years_data.append(yearly_data)

all_years_data = torch.stack(all_years_data)

global_mean = all_years_data.mean(dim=(0, 1))
global_std  = all_years_data.std(dim=(0, 1)) + 1e-6

global_mean = global_mean.to(device)
global_std  = global_std.to(device)

print("Global Mean :", global_mean)
print("Global Std  :", global_std)


# =========================================================
# Define Spatial Models
# =========================================================

class GCNNet(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):

        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = GCNConv(
            hidden_channels,
            out_channels
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return x


class GATNet(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        heads=4
    ):

        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=heads,
            concat=True
        )

        self.conv2 = GATConv(
            hidden_channels * heads,
            out_channels,
            heads=1,
            concat=False
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return x


class GATv2Net(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        heads=4
    ):

        super().__init__()

        self.conv1 = GATv2Conv(
            in_channels,
            hidden_channels,
            heads=heads
        )

        self.conv2 = GATv2Conv(
            hidden_channels * heads,
            out_channels,
            heads=1,
            concat=False
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return x


# =========================================================
# Model Dictionary
# =========================================================

models_dict = {
    "GCN": GCNNet,
    "GAT": GATNet,
    "GATv2": GATv2Net
}

results = {}


# =========================================================
# Spatial Encoder Benchmarking
# =========================================================

for model_name, model_class in models_dict.items():

    print("\n===================================")
    print(f"Training {model_name}")
    print("===================================")

    mse_list = []

    for seed in seeds:

        print(f"\nSeed : {seed}")

        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        # -------------------------------------------------
        # Initialize Model
        # -------------------------------------------------

        if model_name == "GCN":

            model = model_class(
                in_channels=2,
                hidden_channels=hidden_channels,
                out_channels=2
            ).to(device)

        else:

            model = model_class(
                in_channels=2,
                hidden_channels=hidden_channels,
                out_channels=2,
                heads=heads
            ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate
        )

        # -------------------------------------------------
        # Train Across ALL YEARS
        # -------------------------------------------------

        for epoch in range(epochs):

            model.train()

            optimizer.zero_grad()

            total_loss = 0

            for year in years:

                yearly_data = torch.load(
                    BASE_PATH + f"node_features_{year}.pt",
                    weights_only=False
                )

                yearly_data = torch.nan_to_num(yearly_data)

                yearly_data = (
                    yearly_data - global_mean.cpu()
                ) / global_std.cpu()

                yearly_data = yearly_data.to(device)

                out = model(
                    yearly_data,
                    data.edge_index
                )

                loss = F.mse_loss(
                    out,
                    yearly_data
                )

                total_loss += loss

            total_loss = total_loss / len(years)

            total_loss.backward()

            optimizer.step()

            # ---------------------------------------------
            # Print Loss
            # ---------------------------------------------

            if (epoch + 1) % 100 == 0:

                print(
                    f"Epoch {epoch+1}/{epochs} | "
                    f"Loss : {total_loss.item():.6f}"
                )

            # ---------------------------------------------
            # Save Checkpoints
            # ---------------------------------------------

            if (epoch + 1) % 200 == 0:

                torch.save(
                    model.state_dict(),
                    f"/kaggle/working/{model_name}_seed_{seed}_epoch_{epoch+1}.pt"
                )

                print(
                    f"Checkpoint saved at epoch {epoch+1}"
                )

        # -------------------------------------------------
        # Evaluate Spatial Reconstruction
        # -------------------------------------------------

        model.eval()

        yearly_mse = []

        with torch.no_grad():

            for year in years:

                yearly_data = torch.load(
                    BASE_PATH + f"node_features_{year}.pt",
                    weights_only=False
                )

                yearly_data = torch.nan_to_num(yearly_data)

                yearly_data = (
                    yearly_data - global_mean.cpu()
                ) / global_std.cpu()

                yearly_data = yearly_data.to(device)

                out = model(
                    yearly_data,
                    data.edge_index
                )

                mse = F.mse_loss(
                    out,
                    yearly_data
                ).item()

                yearly_mse.append(mse)

        spatial_mse = np.mean(yearly_mse)

        print(f"Spatial MSE : {spatial_mse:.6f}")

        mse_list.append(spatial_mse)

    # -------------------------------------------------
    # Final Results
    # -------------------------------------------------

    mean_mse = np.mean(mse_list)
    std_mse  = np.std(mse_list)

    results[model_name] = (
        mean_mse,
        std_mse
    )

    print(
        f"\n{model_name} Final : "
        f"{mean_mse:.6f} ± {std_mse:.6f}"
    )


# =========================================================
# Final Comparison
# =========================================================

print("\n===================================")
print("Final Spatial Encoder Comparison")
print("===================================")

for model_name, (mean_mse, std_mse) in results.items():

    print(
        f"{model_name} : "
        f"{mean_mse:.6f} ± {std_mse:.6f}"
    )


# =========================================================
# Save Final Results
# =========================================================

results_serializable = {
    k: [float(v[0]), float(v[1])]
    for k, v in results.items()
}

with open(
    "/kaggle/working/spatial_results.json",
    "w"
) as f:

    json.dump(results_serializable, f)

print("Results saved successfully")


# =========================================================
# Train FINAL GATv2 Model
# =========================================================

print("\n===================================")
print("Training Final GATv2 Model")
print("===================================")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

final_model = GATv2Net(
    in_channels=2,
    hidden_channels=hidden_channels,
    out_channels=2,
    heads=heads
).to(device)

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=learning_rate
)


# =========================================================
# Train Across ALL YEARS
# =========================================================

for epoch in range(epochs):

    final_model.train()

    optimizer.zero_grad()

    total_loss = 0

    for year in years:

        yearly_data = torch.load(
            BASE_PATH + f"node_features_{year}.pt",
            weights_only=False
        )

        yearly_data = torch.nan_to_num(yearly_data)

        yearly_data = (
            yearly_data - global_mean.cpu()
        ) / global_std.cpu()

        yearly_data = yearly_data.to(device)

        out = final_model(
            yearly_data,
            data.edge_index
        )

        loss = F.mse_loss(
            out,
            yearly_data
        )

        total_loss += loss

    total_loss = total_loss / len(years)

    total_loss.backward()

    optimizer.step()

    # -----------------------------------------------------
    # Print Loss
    # -----------------------------------------------------

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss : {total_loss.item():.6f}"
        )

    # -----------------------------------------------------
    # Save Checkpoints
    # -----------------------------------------------------

    if (epoch + 1) % 200 == 0:

        torch.save(
            final_model.state_dict(),
            f"/kaggle/working/final_gatv2_epoch_{epoch+1}.pt"
        )

        print(
            f"Checkpoint saved at epoch {epoch+1}"
        )


# =========================================================
# Save Final Model
# =========================================================

torch.save(
    final_model.state_dict(),
    "/kaggle/working/final_gatv2_model.pt"
)

print("Final GATv2 model saved successfully")


# =========================================================
# Generate Spatial Embeddings
# =========================================================

print("\n===================================")
print("Generating Spatial Embeddings")
print("===================================")

final_model.eval()

with torch.no_grad():

    for year in years:

        print(f"Generating embeddings for year {year}")

        yearly_data = torch.load(
            BASE_PATH + f"node_features_{year}.pt",
            weights_only=False
        )

        yearly_data = torch.nan_to_num(yearly_data)

        yearly_data = (
            yearly_data - global_mean.cpu()
        ) / global_std.cpu()

        yearly_data = yearly_data.to(device)

        out_year = final_model(
            yearly_data,
            data.edge_index
        )

        torch.save(
            out_year.cpu(),
            f"/kaggle/working/spatial_embeddings_{year}.pt"
        )

print("\nAll spatial embeddings saved successfully.")